# group checkin cluster id k5

`0. US_group_checkin_labeled`의 각 체크인 좌표를 `3. loc_cluster_info_k5`에 저장된 5개 클러스터 중심과 비교해 가장 가까운 클러스터 id를 `cluster_id_k`로 저장합니다.

In [17]:
import numpy as np
from pymongo import MongoClient, UpdateOne
from tqdm.auto import tqdm


In [18]:
HOST = "10.255.68.40"
PORT = 27017
DB_NAME = "ejoow"

GROUP_CHECKIN_COL = "0. US_group_checkin_labeled"
CLUSTER_INFO_COL = "3. loc_cluster_info_k5"
ASSIGN_FIELD = "cluster_id_k"
ASSIGN_BATCH = 20000

client = MongoClient(host=HOST, port=PORT)
db = client[DB_NAME]
group_checkin_col = db[GROUP_CHECKIN_COL]
cluster_info_col = db[CLUSTER_INFO_COL]

query = {"Latitude": {"$ne": None}, "Longitude": {"$ne": None}}
n_group_points = group_checkin_col.count_documents(query)

print("db:", DB_NAME)
print("group checkin input:", GROUP_CHECKIN_COL)
print("cluster info input:", CLUSTER_INFO_COL)
print("assign field:", ASSIGN_FIELD)
print("group checkins with coordinates:", n_group_points)


db: ejoow
group checkin input: 0. US_group_checkin_labeled
cluster info input: 3. loc_cluster_info_k5
assign field: cluster_id_k
group checkins with coordinates: 184256


## 1. 그룹 체크인의 위도/경도 좌표 확인

In [19]:
sample_checkins = list(
    group_checkin_col.find(query, {"_id": 1, "Latitude": 1, "Longitude": 1}).limit(5)
)
sample_checkins


[{'_id': ObjectId('699ed7cc0f6b516462485585'),
  'Latitude': 31.188807,
  'Longitude': -81.376461},
 {'_id': ObjectId('699ed7cc0f6b516462485586'),
  'Latitude': 33.639431,
  'Longitude': -111.879551},
 {'_id': ObjectId('699ed7cc0f6b516462485587'),
  'Latitude': 41.953512,
  'Longitude': -87.747146},
 {'_id': ObjectId('699ed7cc0f6b516462485588'),
  'Latitude': 42.367106,
  'Longitude': -71.074312},
 {'_id': ObjectId('699ed7cc0f6b516462485589'),
  'Latitude': 36.204666,
  'Longitude': -86.738245}]

## 2. k=5 클러스터 중심 좌표 로드

In [20]:
cluster_docs = list(
    cluster_info_col.find(
        {},
        {"_id": 1, "cluster_id": 1, "latitude": 1, "longitude": 1},
    ).sort("cluster_id", 1)
)

if len(cluster_docs) != 5:
    raise RuntimeError(f"Expected 5 clusters in {CLUSTER_INFO_COL}, found {len(cluster_docs)}")

cluster_docs


[{'_id': 'k_5_cluster_0',
  'cluster_id': 0,
  'latitude': 40.82884979248047,
  'longitude': -73.99425506591797},
 {'_id': 'k_5_cluster_1',
  'cluster_id': 1,
  'latitude': 40.825687408447266,
  'longitude': -73.99960327148438},
 {'_id': 'k_5_cluster_2',
  'cluster_id': 2,
  'latitude': 40.654666900634766,
  'longitude': -74.00653076171875},
 {'_id': 'k_5_cluster_3',
  'cluster_id': 3,
  'latitude': 40.772438049316406,
  'longitude': -73.8897933959961},
 {'_id': 'k_5_cluster_4',
  'cluster_id': 4,
  'latitude': 40.660621643066406,
  'longitude': -73.78041076660156}]

## 3. 가장 가까운 클러스터 id 계산 함수

In [21]:
cluster_ids = np.asarray([doc["cluster_id"] for doc in cluster_docs], dtype=np.int32)
cluster_coords = np.radians(
    np.asarray([[doc["latitude"], doc["longitude"]] for doc in cluster_docs], dtype=np.float64)
)
earth_radius_km = 6371.0088


def nearest_cluster_ids(batch_xy):
    coords = np.radians(np.asarray(batch_xy, dtype=np.float64))

    lat1 = coords[:, 0][:, None]
    lon1 = coords[:, 1][:, None]
    lat2 = cluster_coords[:, 0][None, :]
    lon2 = cluster_coords[:, 1][None, :]

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    distances = 2.0 * earth_radius_km * np.arcsin(np.sqrt(a))
    nearest_idx = np.argmin(distances, axis=1)
    return cluster_ids[nearest_idx], distances[np.arange(len(nearest_idx)), nearest_idx]


## 4. 가장 가까운 클러스터 id를 `cluster_id_k`로 저장

In [24]:
cursor = group_checkin_col.find(
    query,
    {"_id": 1, "Latitude": 1, "Longitude": 1},
    no_cursor_timeout=True,
)

ops = []
buf_ids = []
buf_xy = []
pbar = tqdm(total=n_group_points, desc=f"Assigning {ASSIGN_FIELD}")

try:
    for doc in cursor:
        buf_ids.append(doc["_id"])
        buf_xy.append([doc["Latitude"], doc["Longitude"]])

        if len(buf_ids) >= ASSIGN_BATCH:
            nearest_ids, _ = nearest_cluster_ids(buf_xy)
            for _id, cluster_id in zip(buf_ids, nearest_ids):
                ops.append(UpdateOne({"_id": _id}, {"$set": {ASSIGN_FIELD: int(cluster_id)}}))

            group_checkin_col.bulk_write(ops, ordered=False)
            pbar.update(len(buf_ids))
            ops.clear()
            buf_ids.clear()
            buf_xy.clear()

    if buf_ids:
        nearest_ids, _ = nearest_cluster_ids(buf_xy)
        for _id, cluster_id in zip(buf_ids, nearest_ids):
            ops.append(UpdateOne({"_id": _id}, {"$set": {ASSIGN_FIELD: int(cluster_id)}}))

        group_checkin_col.bulk_write(ops, ordered=False)
        pbar.update(len(buf_ids))
finally:
    pbar.close()
    cursor.close()

print("done:", ASSIGN_FIELD, "updated in", GROUP_CHECKIN_COL)


Assigning cluster_id_k:   0%|          | 0/184256 [00:00<?, ?it/s]

done: cluster_id_k updated in 0. US_group_checkin_labeled


## 5. 저장 결과 확인

In [25]:
list(
    group_checkin_col.find(
        {ASSIGN_FIELD: {"$exists": True}},
        {"_id": 1, "Latitude": 1, "Longitude": 1, ASSIGN_FIELD: 1},
    ).limit(5)
)


[{'_id': ObjectId('699ed7cc0f6b516462485585'),
  'Latitude': 31.188807,
  'Longitude': -81.376461,
  'cluster_id_k': 2},
 {'_id': ObjectId('699ed7cc0f6b516462485586'),
  'Latitude': 33.639431,
  'Longitude': -111.879551,
  'cluster_id_k': 2},
 {'_id': ObjectId('699ed7cc0f6b516462485587'),
  'Latitude': 41.953512,
  'Longitude': -87.747146,
  'cluster_id_k': 1},
 {'_id': ObjectId('699ed7cc0f6b516462485588'),
  'Latitude': 42.367106,
  'Longitude': -71.074312,
  'cluster_id_k': 3},
 {'_id': ObjectId('699ed7cc0f6b516462485589'),
  'Latitude': 36.204666,
  'Longitude': -86.738245,
  'cluster_id_k': 2}]

In [26]:
assigned_cluster_ids = sorted(group_checkin_col.distinct(ASSIGN_FIELD))
print(f"{GROUP_CHECKIN_COL} {ASSIGN_FIELD} distinct values:", assigned_cluster_ids)
print("count:", len(assigned_cluster_ids))


0. US_group_checkin_labeled cluster_id_k distinct values: [0, 1, 2, 3, 4]
count: 5
